<a href="https://colab.research.google.com/github/shweta-1202/flyrank-ml-internship/blob/main/work/notebooks/w04_signal_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shweta-1202/flyrank-ml-internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

I first looked at the distributions of the main signals used for content refresh. Some fields may have heavy tails, meaning that a small number of pages have much larger values than most pages. This is important because extreme values can affect averages and model results.

In [1]:
import os
import subprocess
import pandas as pd
import numpy as np

if not os.path.exists("flyrank-ml-internship-starter"):
    subprocess.run([
        "git", "clone",
        "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
    ], check=True)

df = pd.read_csv(
    "flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv"
)

signals = [
    "impressions_90d",
    "avg_position",
    "ctr",
    "content_age_days",
    "word_count"
]

df[signals].describe()

,impressions_90d,avg_position,ctr,content_age_days,word_count
count,30000.000000,30000.00000,30000.000000,30000.00000,22301.000000
mean,5200.366300,16.34238,0.510733,256.16780,3107.760325
std,16838.019547,15.21679,3.279162,132.70793,1452.382598
min,1.000000,0.00000,0.000000,90.00000,8.000000
25%,81.000000,6.20000,0.000000,132.00000,2413.000000
50%,731.000000,10.80000,0.070000,236.00000,2877.000000
75%,3615.250000,22.30000,0.290000,333.00000,3666.000000
max,517715.000000,245.00000,100.000000,564.00000,9546.000000


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

In [2]:
df["is_declining"] = (
    df["trend_direction"].str.lower() == "down"
).astype(int)

print(df["is_declining"].value_counts())

is_declining
1    16262
0    13738
Name: count, dtype: int64


### Signal 1: Content age

I expected older pages to have a higher chance of declining because older content may need more frequent review. I compare the declining rate for newer and older pages. The result is an observed relationship, not proof that age causes decline.

In [3]:
age_groups = pd.qcut(
    df["content_age_days"],
    q=4,
    duplicates="drop"
)

age_result = df.groupby(
    age_groups,
    observed=False
)["is_declining"].mean()

print(age_result)

content_age_days
(89.999, 132.0]    0.600027
(132.0, 236.0]     0.639764
(236.0, 333.0]     0.493856
(333.0, 564.0]     0.421541
Name: is_declining, dtype: float64


### Signal 2: Search impressions

I checked whether pages with different levels of search impressions have different declining rates. I expected visibility to provide useful information for deciding which pages deserve attention.

In [4]:
impression_groups = pd.qcut(
    df["impressions_90d"],
    q=4,
    duplicates="drop"
)

impression_result = df.groupby(
    impression_groups,
    observed=False
)["is_declining"].mean()

print(impression_result)

impressions_90d
(0.999, 81.0]          0.376116
(81.0, 731.0]          0.604614
(731.0, 3615.25]       0.625634
(3615.25, 517715.0]    0.562000
Name: is_declining, dtype: float64


### Signal 3: Average search position

I checked whether average search position is related to the declining label. Pages with weaker search positions may be more likely to need attention, but I will treat the result as directional rather than causal.

In [5]:
position_groups = pd.qcut(
    df["avg_position"],
    q=4,
    duplicates="drop"
)

position_result = df.groupby(
    position_groups,
    observed=False
)["is_declining"].mean()

print(position_result)

avg_position
(-0.001, 6.2]    0.461355
(6.2, 10.8]      0.579904
(10.8, 22.3]     0.610694
(22.3, 245.0]    0.516821
Name: is_declining, dtype: float64


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

### Flag-linked signal: CTR

I checked CTR because it is a useful search-performance signal for content decisions. The test compares declining rates across CTR groups. This checks whether the data supports using CTR as a directional signal for prioritization.

In [6]:
ctr_groups = pd.qcut(
    df["ctr"],
    q=4,
    duplicates="drop"
)

ctr_result = df.groupby(
    ctr_groups,
    observed=False
)["is_declining"].mean()

print(ctr_result)

ctr
(-0.001, 0.07]    0.523910
(0.07, 0.29]      0.604825
(0.29, 100.0]     0.515331
Name: is_declining, dtype: float64


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

The signal audit helps the content team understand which available signals may be useful for prioritizing pages for review. The results should be used as directional decision-support rather than as proof that a particular signal causes a page to decline. Signals that show a consistent relationship can be considered together when building a refresh-priority ranking.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.